# WarpLab v1
Self-optimizing workflows for CUDA kernels.

## Section 0 — Environment Setup (Optional for Colab/Kaggle)

In [ ]:
import sys, os
if 'google.colab' in sys.modules or os.path.exists('/kaggle/working'):
    print("Detected Colab/Kaggle environment. Installing dependencies...")
    !pip install pydantic pandas numpy jinja2 matplotlib pyyaml plotly ipywidgets
    # If running from a git clone, install the local package
    if os.path.exists('warplab'):
        !pip install -e .
    else:
        print("Warning: warplab package not found in current directory.")

## Section 1 — Setup

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from warplab.config import load_project_config
from warplab.env import get_env_fingerprint
from warplab.memory import Memory
from warplab.utils import generate_id, get_kernel_signature
from warplab.compiler import compile_kernel
from warplab.validator import run_validator
from warplab.benchmark import run_benchmark
from warplab.search import generate_random_candidates, generate_prior_guided_candidates
from warplab.scoring import score_candidate
from warplab.report import generate_markdown_report
from warplab.profiler import run_profiler, BottleneckInference
from warplab.visualize import plot_performance_distribution, plot_parameter_impact
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

# Set root paths
# If in Colab/Kaggle, the root might be absolute to avoid path confusion
ROOT_DIR = Path(".").resolve() if 'google.colab' in sys.modules or os.path.exists('/kaggle/working') else Path("..").resolve()

PROJECTS_DIR = ROOT_DIR / "projects"
RUNS_DIR = ROOT_DIR / "runs"
REPORTS_DIR = ROOT_DIR / "reports"
DB_PATH = ROOT_DIR / "db" / "warplab.sqlite"

RUNS_DIR.mkdir(exist_ok=True, parents=True)
REPORTS_DIR.mkdir(exist_ok=True, parents=True)
DB_PATH.parent.mkdir(exist_ok=True, parents=True)

memory = Memory(DB_PATH)
print(f"WarpLab initialized at {ROOT_DIR}")


## Section 2 — Environment Fingerprint

In [ ]:
fingerprint = get_env_fingerprint()
pd.Series(fingerprint).to_frame("Value")

## Section 3 — Baseline Compile

In [ ]:
project_name = "saxpy"
project_path = PROJECTS_DIR / project_name
config = load_project_config(project_path)

baseline_bin = project_path / "build" / "bench"
validate_bin = project_path / "build" / "validate"
project_path.joinpath("build").mkdir(exist_ok=True, parents=True)

print(f"Compiling baseline for {project_name}...")
res_bench = compile_kernel(project_path, config.build["compile_kernel"], baseline_bin)
res_valid = compile_kernel(project_path, config.build["compile_validate"], validate_bin)

if res_bench.success and res_valid.success:
    print("Baseline compilation successful.")
else:
    print("Baseline compilation failed.")
    print(res_bench.stderr)
    print(res_valid.stderr)

## Section 4 — Baseline Correctness

In [ ]:
validate_cmd = config.run["validate"].format(size=config.input["size"])
v_res = run_validator(validate_bin, validate_cmd)

if v_res.valid:
    print(f"Baseline validation PASSED. Max abs diff: {v_res.data.get('max_abs_diff')}")
else:
    print("Baseline validation FAILED.")
    print(v_res.raw_output)

## Section 5 — Baseline Benchmark & Profile

In [ ]:
bench_cmd = config.run["benchmark"].format(size=config.input["size"], repeats=config.input["repeats"])
b_res = run_benchmark(baseline_bin, bench_cmd)
baseline_latency = b_res.median_ms
print(f"Baseline Median Latency: {baseline_latency:.4f} ms (CV: {b_res.cv*100:.2f}%)")

print("Profiling baseline...")
baseline_metrics = run_profiler(baseline_bin, bench_cmd)
inference = BottleneckInference(baseline_metrics)
baseline_diagnosis = inference.diagnose()
print(f"Baseline Bottleneck Diagnosis: {baseline_diagnosis['diagnosis']}")
print(f"Explanation: {baseline_diagnosis['explanation']}")
print(f"Virtual Mentor Suggestions:\n{baseline_diagnosis['suggestions']}")


## Section 6 — Search Space and Priors

In [ ]:
sig = get_kernel_signature(project_name, project_path / "kernel.cu", str(config.input["size"]))
priors = memory.get_priors(sig, fingerprint["gpu_name"])

if priors:
    print(f"Found {len(priors)} priors for this kernel signature.")
    candidates = generate_prior_guided_candidates(config, priors, count=30)
else:
    print("No priors found. Sampling random candidates.")
    candidates = generate_random_candidates(config, count=30)

print(f"Generated {len(candidates)} valid candidate configs.")

## Section 7 — Candidate Evaluation Loop

In [ ]:
run_id = generate_id()
memory.insert_run(run_id, project_name, fingerprint, config.objective)

results = []

def evaluate_candidate(i, cand):
    cand_id = f"{run_id}_{i}"
    flags = cand.to_compile_flags()
    
    cand_bin = RUNS_DIR / f"bench_{cand_id}"
    c_res = compile_kernel(project_path, config.build["compile_kernel"], cand_bin, flags)
    
    metrics = {"compile_success": c_res.success, "validate_success": False, "benchmark_success": False}
    
    if c_res.success:
        v_cand = run_validator(cand_bin, config.run["validate"].format(size=config.input["size"]))
        metrics["validate_success"] = v_cand.valid
        
        if v_cand.valid:
            b_cand = run_benchmark(cand_bin, config.run["benchmark"].format(size=config.input["size"], repeats=10))
            metrics["benchmark_success"] = True
            metrics["latency_ms"] = b_cand.median_ms
            metrics["std_ms"] = b_cand.std_ms
            metrics["cv"] = b_cand.cv
            metrics["speedup"] = baseline_latency / b_cand.median_ms
            metrics["score"] = score_candidate(metrics["speedup"], b_cand.cv, True, True)

    memory.insert_candidate(cand_id, run_id, cand.json(), metrics)
    return {**metrics, "config": cand.dict(), "flags": flags, "id": cand_id}

print(f"Compiling and evaluating {len(candidates)} candidates in parallel...")
with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
    futures = [executor.submit(evaluate_candidate, i, cand) for i, cand in enumerate(candidates)]
    for future in concurrent.futures.as_completed(futures):
        res = future.result()
        if res.get("benchmark_success"):
            print(f"[{res['id']}] SUCCESS | latency: {res['latency_ms']:.4f}ms | speedup: {res['speedup']:.2f}x | flags: {res['flags']}")
        else:
            print(f"[{res['id']}] FAILED  | flags: {res['flags']}")
        results.append(res)

df_results = pd.DataFrame(results).dropna(subset=["score"]).sort_values("score", ascending=False)
display(df_results.head(10))


## Section 8 — Memory Update & Local Refinement

In [ ]:
if not df_results.empty:
    best_row = df_results.iloc[0]
    best_config_json = json.dumps(best_row["config"])
    memory.update_priors(sig, fingerprint["gpu_name"], best_config_json, best_row["score"])
    print(f"Priors updated for {project_name}.")

## Section 9 — Final Report

In [ ]:
if not df_results.empty:
    best_cand = df_results.iloc[0].to_dict()
    baseline_metrics = {"latency_ms": baseline_latency, "cv": b_res.cv}
    report_file = generate_markdown_report(run_id, project_name, fingerprint, baseline_metrics, best_cand, baseline_diagnosis, REPORTS_DIR)
    print(f"Report generated at: {report_file}")


## Section 10 — Visualizations and Learning
WarpLab helps you learn how your configuration parameters affect performance.
See the distribution of variants across the search space to understand latency improvements.

In [ ]:
if not df_results.empty:
    plot_performance_distribution(df_results)
    
    # Plot the first dynamic parameter's impact (example)
    if len(config.search_space.keys()) > 0:
        first_param = list(config.search_space.keys())[0]
        plot_parameter_impact(df_results, first_param)
